In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # JobFlow AI — Transformação Gold
# MAGIC
# MAGIC Este notebook cria a tabela Gold de vagas.
# MAGIC
# MAGIC A Gold é a camada de consumo do projeto:
# MAGIC
# MAGIC - app frontend
# MAGIC - agente de IA
# MAGIC - ranking de vagas
# MAGIC - busca semântica / RAG
# MAGIC - futuras integrações com Lakebase

# COMMAND ----------

from datetime import datetime, timezone

from pyspark.sql import functions as F

# COMMAND ----------

CATALOG = "workspace"
SCHEMA = "jobflow_ai"

SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_remoteok_jobs"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_job_postings"

print("=" * 70)
print("JOBFLOW AI — TRANSFORMAÇÃO GOLD")
print("=" * 70)
print(f"Silver table: {SILVER_TABLE}")
print(f"Gold table: {GOLD_TABLE}")
print(f"Horário UTC: {datetime.now(timezone.utc).isoformat()}")
print("=" * 70)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Carregar Silver

# COMMAND ----------

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

silver_df = spark.table(SILVER_TABLE)

print(f"Registros na Silver: {silver_df.count()}")

display(
    silver_df.select(
        "source_system",
        "source_job_id",
        "company",
        "position",
        "location",
        "salary_min",
        "salary_max",
        "is_remote",
        "has_salary",
    ).limit(10)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Criar tabela Gold padronizada

# COMMAND ----------

gold_df = (
    silver_df
    .withColumn(
        "job_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("source_system"),
                F.col("source_job_id"),
            ),
            256,
        ),
    )
    .withColumn(
        "source_job_key",
        F.concat_ws(
            ":",
            F.col("source_system"),
            F.col("source_job_id"),
        ),
    )
    .withColumnRenamed("position", "job_title")
    .withColumnRenamed("company", "company_name")
    .withColumnRenamed("location", "job_location")
    .withColumnRenamed("description_text", "job_description")
    .withColumnRenamed("search_text", "job_search_text")
    .withColumn(
        "employment_type",
        F.lit(None).cast("string"),
    )
    .withColumn(
        "remote_type",
        F.when(F.col("is_remote") == True, F.lit("remote"))
        .otherwise(F.lit("unknown")),
    )
    .withColumn(
        "salary_currency",
        F.lit("USD"),
    )
    .withColumn(
        "published_at",
        F.to_timestamp(F.col("published_at_raw")),
    )
    .withColumn(
        "is_active",
        F.lit(True),
    )
    .withColumn(
        "data_quality_score",
        (
            F.when(F.col("source_job_id").isNotNull(), F.lit(1)).otherwise(F.lit(0))
            + F.when(F.col("job_title").isNotNull() & (F.col("job_title") != ""), F.lit(1)).otherwise(F.lit(0))
            + F.when(F.col("company_name").isNotNull() & (F.col("company_name") != ""), F.lit(1)).otherwise(F.lit(0))
            + F.when(F.col("job_description").isNotNull() & (F.col("job_description") != ""), F.lit(1)).otherwise(F.lit(0))
            + F.when(F.col("job_url").isNotNull() | F.col("apply_url").isNotNull(), F.lit(1)).otherwise(F.lit(0))
        ) / F.lit(5.0),
    )
    .withColumn(
        "gold_processed_at",
        F.current_timestamp(),
    )
    .select(
        "job_id",
        "source_system",
        "source_job_id",
        "source_job_key",
        "slug",
        "job_title",
        "position_normalized",
        "company_name",
        "company_normalized",
        "job_location",
        "location_normalized",
        "remote_type",
        "employment_type",
        "published_at_raw",
        "published_at",
        "tags_array",
        "tags_text",
        "job_description",
        "job_search_text",
        "salary_min",
        "salary_max",
        "salary_midpoint",
        "salary_currency",
        "has_salary",
        "is_remote",
        "is_active",
        "data_quality_score",
        "apply_url",
        "job_url",
        "payload_json",
        "run_id",
        "ingested_at",
        "file_path",
        "silver_processed_at",
        "gold_processed_at",
    )
)

display(gold_df.limit(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Gravar Gold

# COMMAND ----------

(
    gold_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)

print(f"OK: tabela Gold gravada em {GOLD_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Validações da Gold

# COMMAND ----------

gold_table_df = spark.table(GOLD_TABLE)

summary_df = gold_table_df.agg(
    F.count("*").alias("records"),
    F.countDistinct("job_id").alias("distinct_job_ids"),
    F.countDistinct("company_name").alias("distinct_companies"),
    F.sum(F.when(F.col("job_description") != "", 1).otherwise(0)).alias("records_with_description"),
    F.sum(F.when(F.col("has_salary") == True, 1).otherwise(0)).alias("records_with_salary"),
    F.sum(F.when(F.col("is_remote") == True, 1).otherwise(0)).alias("records_remote"),
    F.round(F.avg("data_quality_score"), 3).alias("avg_data_quality_score"),
)

display(summary_df)

quality_df = gold_table_df.select(
    F.sum(F.when(F.col("job_id").isNull(), 1).otherwise(0)).alias("missing_job_id"),
    F.sum(F.when(F.col("source_job_id").isNull(), 1).otherwise(0)).alias("missing_source_job_id"),
    F.sum(F.when(F.col("job_title").isNull() | (F.col("job_title") == ""), 1).otherwise(0)).alias("missing_job_title"),
    F.sum(F.when(F.col("company_name").isNull() | (F.col("company_name") == ""), 1).otherwise(0)).alias("missing_company"),
    F.sum(F.when(F.col("job_description").isNull() | (F.col("job_description") == ""), 1).otherwise(0)).alias("missing_description"),
    F.sum(F.when(F.col("salary_min") > F.col("salary_max"), 1).otherwise(0)).alias("invalid_salary_range"),
)

display(quality_df)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Exemplos para revisão

# COMMAND ----------

display(
    gold_table_df.select(
        "job_id",
        "source_system",
        "job_title",
        "company_name",
        "job_location",
        "remote_type",
        "tags_text",
        "salary_min",
        "salary_max",
        "data_quality_score",
        F.substring("job_description", 1, 700).alias("description_preview"),
    )
    .orderBy(F.col("gold_processed_at").desc())
    .limit(10)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Empresas com mais vagas

# COMMAND ----------

display(
    gold_table_df.groupBy("company_name")
    .agg(F.count("*").alias("jobs"))
    .orderBy(F.col("jobs").desc(), F.col("company_name"))
    .limit(20)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Tags mais frequentes

# COMMAND ----------

tags_df = (
    gold_table_df
    .select(F.explode_outer("tags_array").alias("tag"))
    .where(F.col("tag").isNotNull())
    .groupBy(F.lower(F.col("tag")).alias("tag"))
    .agg(F.count("*").alias("jobs"))
    .orderBy(F.col("jobs").desc(), F.col("tag"))
)

display(tags_df.limit(30))

# COMMAND ----------

print()
print("=" * 70)
print("RESULTADO: TRANSFORMAÇÃO GOLD CONCLUÍDA")
print("=" * 70)
print(f"gold table: {GOLD_TABLE}")
print(f"records: {gold_table_df.count()}")
print("=" * 70)